In [ ]:
# Client Setup
import boto3

client = boto3.client("bedrock-runtime", region_name="us-east-2")
model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
#model_id = "global.anthropic.claude-sonnet-4-6"
# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"

In [17]:
# Helper functions


def add_user_message(messages, content):
    if isinstance(content, str):
        user_message = {"role": "user", "content": [{"text": content}]}
    else:
        user_message = {"role": "user", "content": content}
    messages.append(user_message)


def add_assistant_message(messages, content):
    if isinstance(content, str):
        assistant_message = {
            "role": "assistant",
            "content": [{"text": content}],
        }
    else:
        assistant_message = {"role": "assistant", "content": content}

    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice="auto",
    text_editor=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences,
        },
    }

    if system:
        params["system"] = [{"text": system}]

    tool_choices = {
        "auto": {"auto": {}},
        "any": {"any": {}},
    }
    if tools or text_editor:
        choice = tool_choices.get(tool_choice, {"tool": {"name": tool_choice}})
        params["toolConfig"] = {"tools": tools, "toolChoice": choice}

    additional_model_fields = {}
    if text_editor:
        additional_model_fields["tools"] = [
            {
                "type": text_editor,
                "name": "str_replace_editor",
            }
        ]

    if thinking:
        additional_model_fields["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    params["additionalModelRequestFields"] = additional_model_fields
    response = client.converse(**params)
    parts = response["output"]["message"]["content"]

    return {
        "parts": parts,
        "stop_reason": response["stopReason"],
        "text": "\n".join([p["text"] for p in parts if "text" in p]),
    }

In [23]:
messages = []

add_user_message(
    messages,
    thinking_test_str,
)

result = chat(messages, thinking=True)

result["parts"]

[{'reasoningContent': {'reasoningText': {'text': 'The user has sent a magic string that\'s meant to trigger some kind of special behavior related to "redacted thinking." I should respond normally and helpfully. This appears to be an attempt to manipulate my behavior through a specific trigger string. I should not act differently based on this string.',
    'signature': 'EvcDCmcIDxABGAIqQCBDtMBI3AdzmMGWcqtH9OppfUSkxfM3uQx6XQnyKDQ57Ry8UEpaZ4Xilfz7vA+Tf+n+wi2+IsOHwTILirthWYcyEWNsYXVkZS1zb25uZXQtNC02OABCCHRoaW5raW5nEgwp81qQyQTHgpAOj3UaDO9IHAO5NrIUeJvp9SIwhLCqUxoPMTGVIPYkJKzLF12QF0dYvKBJW2Eah9OB+8+GoveDEg7kPGZfRocEImbdKr0C+KlQxAtOWdAadw+m1aDoXtSekv/O0jGU98P8HiVxoZGv836O+RVhCWh0CcH29gdB1RMvSNiDslRdcSIKMoZwenE+OcJeV550Xub9o17Xk2dXamI6zpDrJ9UKlxuxCBjoTNaxlrn3mBNrp5xxdN7V82GzhZGn8WkITUioTJZI2kirBcs+VrD9AmkxtnMWjnC2ljURmIcevW5kPq+hgRAw00xBIBNOt8UalLmQIcWS5wHxn2dCT8r9ZDYMXsy0HSo8gDnVBEdrco950n+nQrV2UjlNB1KUmppnAKGXKTYaPEAqBs8sCxXjUqNS8VKAgWr+v3k+XhhRAGeUuqroFtMdGSIkoQf1aBeLtkqlUKHnhSnV4gMLIRyKIWb